In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import glob
import gc
import os
import duckdb
from tqdm import tqdm

In [118]:
df = pd.read_csv("D:/UEA/Business Analytics/P1/Code_Dataset/Data_1/epd_snomed_202512.csv")


In [119]:
df.head(5)

,YEAR_MONTH,REGIONAL_OFFICE_NAME,REGIONAL_OFFICE_CODE,ICB_NAME,ICB_CODE,PCO_NAME,PCO_CODE,PRACTICE_NAME,PRACTICE_CODE,ADDRESS_1,...,BNF_PRESENTATION_NAME,BNF_CHAPTER_PLUS_CODE,QUANTITY,ITEMS,TOTAL_QUANTITY,ADQ_USAGE,NIC,ACTUAL_COST,UNIDENTIFIED,SNOMED_CODE
0,2025-12,LONDON,Y56,NHS NORTH EAST LONDON INTEGRATED CARE BOARD,QMF,NHS NORTH EAST LONDON ICB - A3A8R,A3A8R,BALFOUR ROAD SURGERY,F86042,92 BALFOUR ROAD,...,Mepore dressing 7cm x 8cm,20: Dressings,10.0,2,20.0,0.0,2.40,2.18802,N,1.841100e+13
1,2025-12,LONDON,Y56,NHS NORTH EAST LONDON INTEGRATED CARE BOARD,QMF,NHS NORTH EAST LONDON ICB - A3A8R,A3A8R,BARKING MEDICAL GROUP PRACTICE,F82018,BARKING GROUP PRACTICE,...,Allevyn Adhesive dressing 10cm x 10cm square,20: Dressings,60.0,1,60.0,0.0,145.20,130.88721,N,3.725111e+15
2,2025-12,LONDON,Y56,NHS NORTH EAST LONDON INTEGRATED CARE BOARD,QMF,NHS NORTH EAST LONDON ICB - A3A8R,A3A8R,BALFOUR ROAD SURGERY,F86042,92 BALFOUR ROAD,...,Independence Xtra large wound drainage bag wit...,20: Dressings,30.0,1,30.0,0.0,408.24,408.24000,N,2.274781e+16
3,2025-12,LONDON,Y56,NHS NORTH EAST LONDON INTEGRATED CARE BOARD,QMF,NHS NORTH EAST LONDON ICB - A3A8R,A3A8R,BALFOUR ROAD SURGERY,F86042,92 BALFOUR ROAD,...,Independence Xtra large wound drainage bag wit...,20: Dressings,40.0,1,40.0,0.0,544.32,544.32000,N,2.274781e+16
4,2025-12,LONDON,Y56,NHS NORTH EAST LONDON INTEGRATED CARE BOARD,QMF,NHS NORTH EAST LONDON ICB - A3A8R,A3A8R,BARKING MEDICAL GROUP PRACTICE,F82018,BARKING GROUP PRACTICE,...,Advazorb Plus (T) tracheostomy dressing 8cm x 8cm,20: Dressings,90.0,1,90.0,0.0,105.30,105.30000,N,2.942391e+16


In [121]:
df = df.rename(columns={
    'REGIONAL_OFFICE_NAME': 'REGIONAL_OFFICE',
    'BNF_CHAPTER_PLUS_CODE': 'BNF_CHAPTER_CODE',
    'BNF_CHEMICAL_SUBSTANCE': 'DRUG_NAME',
    'BNF_PRESENTATION_NAME': 'DRUG_DESCRIPTION',
    'ADQUSAGE': 'ADQ_USAGE'
})

df.columns

Index(['YEAR_MONTH', 'REGIONAL_OFFICE', 'REGIONAL_OFFICE_CODE', 'ICB_NAME',
       'ICB_CODE', 'PCO_NAME', 'PCO_CODE', 'PRACTICE_NAME', 'PRACTICE_CODE',
       'ADDRESS_1', 'ADDRESS_2', 'ADDRESS_3', 'ADDRESS_4', 'POSTCODE',
       'BNF_CHEMICAL_SUBSTANCE_CODE', 'DRUG_NAME', 'BNF_PRESENTATION_CODE',
       'DRUG_DESCRIPTION', 'BNF_CHAPTER_CODE', 'QUANTITY', 'ITEMS',
       'TOTAL_QUANTITY', 'ADQ_USAGE', 'NIC', 'ACTUAL_COST', 'UNIDENTIFIED',
       'SNOMED_CODE'],
      dtype='str')

In [122]:
keep_columns = [
    'YEAR_MONTH', 'REGIONAL_OFFICE', 'ICB_NAME', 'ICB_CODE',
    'PRACTICE_NAME', 'PRACTICE_CODE', 'POSTCODE',
    'BNF_CHAPTER_CODE', 'DRUG_NAME',
    'DRUG_DESCRIPTION', 'ITEMS', 'TOTAL_QUANTITY',
    'ADQ_USAGE', 'NIC', 'ACTUAL_COST', 'UNIDENTIFIED'
]

df = df[keep_columns]

In [123]:
df.columns

Index(['YEAR_MONTH', 'REGIONAL_OFFICE', 'ICB_NAME', 'ICB_CODE',
       'PRACTICE_NAME', 'PRACTICE_CODE', 'POSTCODE', 'BNF_CHAPTER_CODE',
       'DRUG_NAME', 'DRUG_DESCRIPTION', 'ITEMS', 'TOTAL_QUANTITY', 'ADQ_USAGE',
       'NIC', 'ACTUAL_COST', 'UNIDENTIFIED'],
      dtype='str')

In [124]:
df['YEAR_MONTH'] = df['YEAR_MONTH'].astype(str)
df['YEAR_MONTH'] = df['YEAR_MONTH'].apply(
    lambda x: x[:4] + '-' + x[4:] if '-' not in x else x
)

In [125]:
print(df[['YEAR_MONTH']].tail())

         YEAR_MONTH
18452669    2025-12
18452670    2025-12
18452671    2025-12
18452672    2025-12
18452673    2025-12


In [126]:
df['ADQ_USAGE'] = df['ADQ_USAGE'].round(2)
df['NIC'] = df['NIC'].round(2)
df['ACTUAL_COST'] = df['ACTUAL_COST'].round(2)

In [127]:
print(df[['ADQ_USAGE', 'NIC', 'ACTUAL_COST']].tail())

          ADQ_USAGE    NIC  ACTUAL_COST
18452669        0.0   2.75         2.49
18452670        0.0  26.86        24.22
18452671        0.0  26.86        24.22
18452672        0.0   2.69         2.44
18452673        0.0  98.22        98.22


In [128]:
df['TOTAL_QUANTITY'] = df['TOTAL_QUANTITY'].astype(int)
print(df[['TOTAL_QUANTITY']].head())

   TOTAL_QUANTITY
0              20
1              60
2              30
3              40
4              90


In [129]:
print(df.head());

  YEAR_MONTH REGIONAL_OFFICE                                     ICB_NAME  \
0    2025-12          LONDON  NHS NORTH EAST LONDON INTEGRATED CARE BOARD   
1    2025-12          LONDON  NHS NORTH EAST LONDON INTEGRATED CARE BOARD   
2    2025-12          LONDON  NHS NORTH EAST LONDON INTEGRATED CARE BOARD   
3    2025-12          LONDON  NHS NORTH EAST LONDON INTEGRATED CARE BOARD   
4    2025-12          LONDON  NHS NORTH EAST LONDON INTEGRATED CARE BOARD   

  ICB_CODE                   PRACTICE_NAME PRACTICE_CODE  POSTCODE  \
0      QMF            BALFOUR ROAD SURGERY        F86042   IG1 4JE   
1      QMF  BARKING MEDICAL GROUP PRACTICE        F82018  IG11 9LT   
2      QMF            BALFOUR ROAD SURGERY        F86042   IG1 4JE   
3      QMF            BALFOUR ROAD SURGERY        F86042   IG1 4JE   
4      QMF  BARKING MEDICAL GROUP PRACTICE        F82018  IG11 9LT   

  BNF_CHAPTER_CODE                               DRUG_NAME  \
0    20: Dressings      Wound Management & Other Dress

In [130]:
df['COST_PER_ITEM'] = (df['ACTUAL_COST'] / df['ITEMS']).round(2)
df['NIC_RATIO'] = (df['ACTUAL_COST'] / df['NIC']).round(2)
df['MONTH'] = df['YEAR_MONTH'].str[5:].astype(int)
df['QUARTER'] = df['MONTH'].apply(lambda x: (x - 1) // 3 + 1)

In [131]:
print(df.head());

  YEAR_MONTH REGIONAL_OFFICE                                     ICB_NAME  \
0    2025-12          LONDON  NHS NORTH EAST LONDON INTEGRATED CARE BOARD   
1    2025-12          LONDON  NHS NORTH EAST LONDON INTEGRATED CARE BOARD   
2    2025-12          LONDON  NHS NORTH EAST LONDON INTEGRATED CARE BOARD   
3    2025-12          LONDON  NHS NORTH EAST LONDON INTEGRATED CARE BOARD   
4    2025-12          LONDON  NHS NORTH EAST LONDON INTEGRATED CARE BOARD   

  ICB_CODE                   PRACTICE_NAME PRACTICE_CODE  POSTCODE  \
0      QMF            BALFOUR ROAD SURGERY        F86042   IG1 4JE   
1      QMF  BARKING MEDICAL GROUP PRACTICE        F82018  IG11 9LT   
2      QMF            BALFOUR ROAD SURGERY        F86042   IG1 4JE   
3      QMF            BALFOUR ROAD SURGERY        F86042   IG1 4JE   
4      QMF  BARKING MEDICAL GROUP PRACTICE        F82018  IG11 9LT   

  BNF_CHAPTER_CODE                               DRUG_NAME  \
0    20: Dressings      Wound Management & Other Dress

In [132]:
print(df[df['UNIDENTIFIED'] == 'Y'].shape[0])

17165


In [133]:
df[df['UNIDENTIFIED'] == 'Y']

,YEAR_MONTH,REGIONAL_OFFICE,ICB_NAME,ICB_CODE,PRACTICE_NAME,PRACTICE_CODE,POSTCODE,BNF_CHAPTER_CODE,DRUG_NAME,DRUG_DESCRIPTION,ITEMS,TOTAL_QUANTITY,ADQ_USAGE,NIC,ACTUAL_COST,UNIDENTIFIED,COST_PER_ITEM,NIC_RATIO,MONTH,QUARTER
205025,2025-12,NORTH EAST AND YORKSHIRE,NHS NORTH EAST AND NORTH CUMBRIA INTEGRATED CA...,QHM,UNIDENTIFIED DOCTORS,00N998,NE32 5NN,20: Dressings,Wound Management & Other Dressings,Tegaderm Film dressing 6cm x 7cm,1,4,0.0,1.64,1.49,Y,1.49,0.91,12,4
205648,2025-12,NORTH EAST AND YORKSHIRE,NHS NORTH EAST AND NORTH CUMBRIA INTEGRATED CA...,QHM,UNIDENTIFIED DOCTORS,00N998,NE32 5NN,21: Appliances,Emollients,Hydromol Bath & Shower emollient,1,1000,0.0,9.68,8.74,Y,8.74,0.90,12,4
205649,2025-12,NORTH EAST AND YORKSHIRE,NHS NORTH EAST AND NORTH CUMBRIA INTEGRATED CA...,QHM,UNIDENTIFIED DOCTORS,00N998,NE32 5NN,21: Appliances,Emollients,Zeroderm ointment,1,500,0.0,4.35,3.93,Y,3.93,0.90,12,4
205650,2025-12,NORTH EAST AND YORKSHIRE,NHS NORTH EAST AND NORTH CUMBRIA INTEGRATED CA...,QHM,UNIDENTIFIED DOCTORS,00N998,NE32 5NN,21: Appliances,Emollients,Hydromol ointment,1,500,0.0,13.20,11.91,Y,11.91,0.90,12,4
206877,2025-12,NORTH EAST AND YORKSHIRE,NHS NORTH EAST AND NORTH CUMBRIA INTEGRATED CA...,QHM,UNIDENTIFIED DOCTORS,00N998,NE32 5NN,01: Gastro-Intestinal System,Lansoprazole,Lansoprazole 30mg gastro-resistant capsules,1,28,42.0,1.02,0.83,Y,0.83,0.81,12,4
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
18360208,2025-12,LONDON,NHS SOUTH EAST LONDON INTEGRATED CARE BOARD,QKK,UNIDENTIFIED DOCTORS,NCN998,NaN,13: Skin,Isotretinoin,Isotretinoin 20mg capsules,1,120,0.0,119.36,95.46,Y,95.46,0.80,12,4
18360209,2025-12,LONDON,NHS SOUTH EAST LONDON INTEGRATED CARE BOARD,QKK,UNIDENTIFIED DOCTORS,NCN998,NaN,13: Skin,Isotretinoin,Isotretinoin 20mg capsules,2,120,0.0,119.36,95.47,Y,47.74,0.80,12,4
18360210,2025-12,LONDON,NHS SOUTH EAST LONDON INTEGRATED CARE BOARD,QKK,UNIDENTIFIED DOCTORS,NCN998,NaN,13: Skin,Isotretinoin,Isotretinoin 20mg capsules,4,360,0.0,358.08,286.39,Y,71.60,0.80,12,4
18360478,2025-12,LONDON,NHS SOUTH EAST LONDON INTEGRATED CARE BOARD,QKK,UNIDENTIFIED DOCTORS,NCN998,NaN,13: Skin,Isotretinoin,Isotretinoin 10mg capsules,1,30,0.0,16.99,13.60,Y,13.60,0.80,12,4


In [134]:
df = df[df['UNIDENTIFIED'] != 'Y']

In [135]:
print(df['UNIDENTIFIED'].value_counts())

UNIDENTIFIED
N    18435509
Name: count, dtype: int64


In [136]:
print(df.isnull().sum())

YEAR_MONTH             0
REGIONAL_OFFICE        0
ICB_NAME               0
ICB_CODE               0
PRACTICE_NAME          0
PRACTICE_CODE          0
POSTCODE            1315
BNF_CHAPTER_CODE       0
DRUG_NAME              0
DRUG_DESCRIPTION       0
ITEMS                  0
TOTAL_QUANTITY         0
ADQ_USAGE              0
NIC                    0
ACTUAL_COST            0
UNIDENTIFIED           0
COST_PER_ITEM          0
NIC_RATIO            329
MONTH                  0
QUARTER                0
dtype: int64


In [137]:
print(df[df['NIC'] == 0].shape[0])

333


In [138]:
before = len(df)
print(f"Rows before dropping: {before}")


Rows before dropping: 18435509


In [139]:
df = df[df['NIC'] != 0]


In [140]:
after = len(df)
print(f"Rows after dropping: {after}")
print(f"Rows dropped: {before - after}")

Rows after dropping: 18435176
Rows dropped: 333


In [141]:
print(f"NIC = 0: {(df['NIC'] == 0).sum()}")
print(f"NIC missing: {df['NIC'].isnull().sum()}")

NIC = 0: 0
NIC missing: 0


In [142]:
df = df.drop(columns=['POSTCODE'])

In [143]:
print(df.head());

  YEAR_MONTH REGIONAL_OFFICE                                     ICB_NAME  \
0    2025-12          LONDON  NHS NORTH EAST LONDON INTEGRATED CARE BOARD   
1    2025-12          LONDON  NHS NORTH EAST LONDON INTEGRATED CARE BOARD   
2    2025-12          LONDON  NHS NORTH EAST LONDON INTEGRATED CARE BOARD   
3    2025-12          LONDON  NHS NORTH EAST LONDON INTEGRATED CARE BOARD   
4    2025-12          LONDON  NHS NORTH EAST LONDON INTEGRATED CARE BOARD   

  ICB_CODE                   PRACTICE_NAME PRACTICE_CODE BNF_CHAPTER_CODE  \
0      QMF            BALFOUR ROAD SURGERY        F86042    20: Dressings   
1      QMF  BARKING MEDICAL GROUP PRACTICE        F82018    20: Dressings   
2      QMF            BALFOUR ROAD SURGERY        F86042    20: Dressings   
3      QMF            BALFOUR ROAD SURGERY        F86042    20: Dressings   
4      QMF  BARKING MEDICAL GROUP PRACTICE        F82018    20: Dressings   

                                DRUG_NAME  \
0      Wound Management & Oth

In [144]:
df.to_csv("D:/UEA/Business Analytics/P1/Code_Dataset/Data_Main/epd_snomed_202512_clean.csv", index=False)

In [146]:
df = pd.read_csv("D:/UEA/Business Analytics/P1/Code_Dataset/Data_Main/epd_snomed_202512_clean.csv")

In [147]:
df.head(5)

,YEAR_MONTH,REGIONAL_OFFICE,ICB_NAME,ICB_CODE,PRACTICE_NAME,PRACTICE_CODE,BNF_CHAPTER_CODE,DRUG_NAME,DRUG_DESCRIPTION,ITEMS,TOTAL_QUANTITY,ADQ_USAGE,NIC,ACTUAL_COST,UNIDENTIFIED,COST_PER_ITEM,NIC_RATIO,MONTH,QUARTER
0,2025-12,LONDON,NHS NORTH EAST LONDON INTEGRATED CARE BOARD,QMF,BALFOUR ROAD SURGERY,F86042,20: Dressings,Wound Management & Other Dressings,Mepore dressing 7cm x 8cm,2,20,0.0,2.40,2.19,N,1.10,0.91,12,4
1,2025-12,LONDON,NHS NORTH EAST LONDON INTEGRATED CARE BOARD,QMF,BARKING MEDICAL GROUP PRACTICE,F82018,20: Dressings,Wound Management & Other Dressings,Allevyn Adhesive dressing 10cm x 10cm square,1,60,0.0,145.20,130.89,N,130.89,0.90,12,4
2,2025-12,LONDON,NHS NORTH EAST LONDON INTEGRATED CARE BOARD,QMF,BALFOUR ROAD SURGERY,F86042,20: Dressings,Wound Management & Other Dressings,Independence Xtra large wound drainage bag wit...,1,30,0.0,408.24,408.24,N,408.24,1.00,12,4
3,2025-12,LONDON,NHS NORTH EAST LONDON INTEGRATED CARE BOARD,QMF,BALFOUR ROAD SURGERY,F86042,20: Dressings,Wound Management & Other Dressings,Independence Xtra large wound drainage bag wit...,1,40,0.0,544.32,544.32,N,544.32,1.00,12,4
4,2025-12,LONDON,NHS NORTH EAST LONDON INTEGRATED CARE BOARD,QMF,BARKING MEDICAL GROUP PRACTICE,F82018,20: Dressings,Tracheostomy & Laryngectomy Appliances,Advazorb Plus (T) tracheostomy dressing 8cm x 8cm,1,90,0.0,105.30,105.30,N,105.30,1.00,12,4
